





























































































































































































































































































































































































































































































# 🎁 Bônus da Semana 11 — CRUD no Python: Integração Python e PostgreSQL

Este notebook é um **bônus opcional**, separado do notebook principal desta semana (`notebook_colab_aluno.ipynb`, que usa `pyodbc` + SQLite).

Aqui você repete exatamente a mesma lógica de conexão — **conectar → criar cursor → executar comandos SQL → encerrar** — mas contra um banco de verdade que você já criou: o **PostgreSQL**, no banco `northwind` (o mesmo que você usa desde a Semana 08 — 91 clientes, 830 pedidos, 77 produtos reais, com clientes do Brasil incluídos).

**⚠️ Pré-requisito:** o PostgreSQL precisa estar rodando na sua máquina, e o banco `northwind` já precisa existir (criado nas Semanas 08/09). Troque `SUA_SENHA_AQUI` pela senha que você mesmo configurou no PostgreSQL.

## Connect — a mesma lógica, biblioteca diferente

Onde no notebook principal você usava `pyodbc.connect(...)` com uma string cheia de `Driver={...}`, o `psycopg2` (a biblioteca oficial do PostgreSQL) usa parâmetros nomeados diretos — mais simples, porque foi feita só para esse banco (não precisa de driver ODBC nenhum).

In [2]:
%pip install psycopg2

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 3.4 MB/s eta 0:00:01
   ------------------ --------------------- 1.3/2.8 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 4.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password=123456,
)
cursor = conexao.cursor()
print("Conexão bem sucedida")


Conexão bem sucedida


## Read — consultando com parâmetro seguro

A diretoria da Northwind quer saber quais clientes são do Brasil. Repare no `%s` dentro do SQL, no lugar do valor `'Brazil'` — é assim que o `psycopg2` recebe parâmetros: o valor vai numa tupla separada, nunca colado direto na string.

**Isso não é o mesmo `%s` de formatação de texto do Python** (tipo `"Olá, %s" % nome`), mesmo parecendo igual. Aqui, o `%s` é um espaço reservado que o `psycopg2` interpreta sozinho, mandando o valor pro banco separado do comando SQL — é exatamente essa separação que impede o SQL Injection (ver abaixo). Se fosse formatação comum do Python, o valor viraria texto colado no comando, e o problema de segurança voltaria a existir.

**Por que a tupla `("Brazil",)` tem uma vírgula depois de um valor só?** Em Python, `("Brazil")` sem vírgula NÃO é uma tupla — é só a string `"Brazil"` entre parênteses (os parênteses aqui não fariam nada). A vírgula depois do valor é o que avisa o Python "isto é uma tupla de 1 item", mesmo parecendo sobrando à primeira vista.

**Por que não colar o valor direto no texto?** Colar valores direto na string de SQL (`f"... WHERE country = '{pais}'"`) abre brecha pra *SQL Injection* — alguém pode digitar um valor malicioso que muda o comando inteiro. Passar o valor como parâmetro (`%s` + tupla) faz o `psycopg2` tratar esse valor sempre como um dado puro, nunca como parte do comando.

In [8]:
cursor.execute("SELECT company_name, city, country FROM customers WHERE country = %s", ("Brazil",))
clientes_brasil = cursor.fetchall()
print(clientes_brasil)

[('Comércio Mineiro', 'Sao Paulo', 'Brazil'), ('Familia Arquibaldo', 'Sao Paulo', 'Brazil'), ('Gourmet Lanchonetes', 'Campinas', 'Brazil'), ('Hanari Carnes', 'Rio de Janeiro', 'Brazil'), ('Que Delícia', 'Rio de Janeiro', 'Brazil'), ('Queen Cozinha', 'Sao Paulo', 'Brazil'), ('Ricardo Adocicados', 'Rio de Janeiro', 'Brazil'), ('Tradição Hipermercados', 'Sao Paulo', 'Brazil'), ('Wellington Importadora', 'Resende', 'Brazil')]


A mesma consulta, agora com `pd.read_sql()` — o resultado já pronto como DataFrame. Uma observação: o Pandas mostra um aviso amarelo (`UserWarning`) dizendo que só testa oficialmente conexões `sqlite3` ou SQLAlchemy — pode ignorar, o resultado sai correto do mesmo jeito.

In [9]:
import pandas as pd

tabela_brasil = pd.read_sql(
    "SELECT company_name, city, country FROM customers WHERE country = %s",
    conexao,
    params=("Brazil",),
)
display(tabela_brasil)

C:\Users\s2bcl\AppData\Local\Temp\ipykernel_11516\55738176.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tabela_brasil = pd.read_sql(


,company_name,city,country
0,Comércio Mineiro,Sao Paulo,Brazil
1,Familia Arquibaldo,Sao Paulo,Brazil
2,Gourmet Lanchonetes,Campinas,Brazil
3,Hanari Carnes,Rio de Janeiro,Brazil
4,Que Delícia,Rio de Janeiro,Brazil
5,Queen Cozinha,Sao Paulo,Brazil
6,Ricardo Adocicados,Rio de Janeiro,Brazil
7,Tradição Hipermercados,Sao Paulo,Brazil
8,Wellington Importadora,Resende,Brazil


## Create — cadastrando um cliente novo

**Contextualização:** a Squad Estúdio virou cliente novo da Northwind e precisa ser cadastrada. Diferente do `chinook.db` (onde o Id era automático), a tabela `customers` do Northwind usa um código de 5 letras escolhido na hora do cadastro (como `'ALFKI'`, `'ANATR'`).

In [10]:
cursor.execute(
    "INSERT INTO customers (customer_id, company_name, contact_name, city, country) VALUES (%s, %s, %s, %s, %s)",
    ("SQUAD", "Squad Estúdio", "Ana Souza", "Curitiba", "Brazil"),
)
conexao.commit()

cursor.execute("SELECT customer_id, company_name, city FROM customers WHERE customer_id = %s", ("SQUAD",))
print(cursor.fetchall())

[('SQUAD', 'Squad Estúdio', 'Curitiba')]


## Update — a Squad Estúdio mudou de cidade

**Contextualização:** a Squad Estúdio avisou que se mudou para São Paulo.

In [11]:
cursor.execute(
    "UPDATE customers SET city = %s WHERE customer_id = %s",("São Paulo", "SQUAD"),
)
conexao.commit()

cursor.execute("SELECT customer_id, city FROM customers WHERE customer_id = %s", ("SQUAD",))
print(cursor.fetchall())

[('SQUAD', 'São Paulo')]


## Um JOIN de verdade — o que o SQLite sozinho não mostrava tão bem

O Northwind tem tabelas de verdade relacionadas entre si (`customers` → `orders` → `order_details` → `products`). Isso permite uma pergunta que nenhuma tabela isolada responde: **quais produtos um cliente específico já comprou?**

Repare que, no código abaixo, cada tabela ganha um **alias** — um apelido curto (`c` para `customers`, `o` para `orders`, `od` para `order_details`, `p` para `products`) — escrito logo depois do nome da tabela (`customers c`, `orders o`). Isso existe só pra não repetir o nome inteiro da tabela toda vez que uma coluna precisa dizer de onde ela vem (`c.company_name`, `o.customer_id`) — numa consulta com 4 tabelas juntas, sem alias o SQL ficaria bem mais longo e repetitivo.

In [12]:
cursor.execute('''
    SELECT c.company_name, p.product_name, od.quantity
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_details od ON od.order_id = o.order_id
    JOIN products p ON p.product_id = od.product_id
    WHERE c.company_name = %s
    ORDER BY p.product_name
''', ("Hanari Carnes",))

print(cursor.fetchall())

[('Hanari Carnes', 'Alice Mutton', 15), ('Hanari Carnes', 'Carnarvon Tigers', 20), ('Hanari Carnes', 'Chartreuse verte', 42), ('Hanari Carnes', 'Côte de Blaye', 5), ('Hanari Carnes', 'Côte de Blaye', 4), ('Hanari Carnes', 'Côte de Blaye', 60), ('Hanari Carnes', 'Filo Mix', 12), ('Hanari Carnes', 'Flotemysost', 9), ('Hanari Carnes', 'Gnocchi di nonna Alice', 20), ('Hanari Carnes', 'Gorgonzola Telino', 20), ('Hanari Carnes', 'Gorgonzola Telino', 35), ('Hanari Carnes', 'Gorgonzola Telino', 10), ('Hanari Carnes', 'Guaraná Fantástica', 35), ('Hanari Carnes', 'Guaraná Fantástica', 35), ('Hanari Carnes', 'Gudbrandsdalsost', 30), ('Hanari Carnes', 'Ikura', 70), ('Hanari Carnes', 'Inlagd Sill', 15), ('Hanari Carnes', 'Inlagd Sill', 25), ('Hanari Carnes', 'Ipoh Coffee', 30), ('Hanari Carnes', "Jack's New England Clam Chowder", 10), ('Hanari Carnes', 'Konbu', 40), ('Hanari Carnes', 'Louisiana Fiery Hot Pepper Sauce', 21), ('Hanari Carnes', 'Louisiana Fiery Hot Pepper Sauce', 36), ('Hanari Carnes'

In [15]:
import pandas as pd
    
sala = """
    SELECT c.company_name, p.product_name, od.quantity
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_details od ON od.order_id = o.order_id
    JOIN products p ON p.product_id = od.product_id
    WHERE c.company_name = %s
    ORDER BY p.product_name
 """
 
df = pd.read_sql_query(sala, conexao, params=("Hanari Carnes",))
print(df)

     company_name                      product_name  quantity
0   Hanari Carnes                      Alice Mutton        15
1   Hanari Carnes                  Carnarvon Tigers        20
2   Hanari Carnes                  Chartreuse verte        42
3   Hanari Carnes                     Côte de Blaye         5
4   Hanari Carnes                     Côte de Blaye         4
5   Hanari Carnes                     Côte de Blaye        60
6   Hanari Carnes                          Filo Mix        12
7   Hanari Carnes                       Flotemysost         9
8   Hanari Carnes            Gnocchi di nonna Alice        20
9   Hanari Carnes                 Gorgonzola Telino        20
10  Hanari Carnes                 Gorgonzola Telino        35
11  Hanari Carnes                 Gorgonzola Telino        10
12  Hanari Carnes                Guaraná Fantástica        35
13  Hanari Carnes                Guaraná Fantástica        35
14  Hanari Carnes                  Gudbrandsdalsost        30
15  Hana

C:\Users\s2bcl\AppData\Local\Temp\ipykernel_11516\3945076497.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sala, conexao, params=("Hanari Carnes",))


## Fato x Dimensão — por que o Northwind é um bom exemplo pra isso

Repare que a última consulta juntou 4 tabelas, e cada uma tem um papel diferente:

| Tabela | Papel | O que ela guarda |
|---|---|---|
| `order_details` | **Fato** | Cada linha é uma transação de verdade: um produto, dentro de um pedido, com quantidade/preço/desconto |
| `customers` | Dimensão | Descreve QUEM comprou |
| `products` | Dimensão | Descreve O QUE foi comprado |
| `orders` | Dimensão (e liga ao cliente) | Descreve QUANDO e para onde foi o pedido |

**Tabela fato** é aquela cujas linhas representam eventos/transações que aconteceram (aqui, cada item vendido) — ela concentra os números que você soma/conta (`quantity`, `unit_price`, `discount`). **Tabelas de dimensão** são as que descrevem, com texto, os "quem/o quê/onde" em volta de cada fato — elas não mudam a cada venda, só são consultadas pra dar contexto ao fato.

Essa distinção é o que torna possível somar receita "por categoria" ou "por país" sem precisar de uma tabela nova pra cada pergunta — você só troca a dimensão que entra no `JOIN`.

### Receita por categoria (Fato + 2 Dimensões)

**Contextualização:** a diretoria quer saber qual categoria de produto gera mais receita, considerando os descontos já aplicados.

A fórmula de receita de cada item é `unit_price × quantity × (1 - discount)` — está em `order_details` (o fato); a categoria de cada produto vem de duas dimensões encadeadas (`products` → `categories`).

Duas coisas novas no código abaixo:
- **`::numeric`** — o PostgreSQL calcula a soma como um número de ponto flutuante, que pode vir com muitas casas decimais estranhas; `::numeric` converte esse resultado pra um tipo numérico exato antes do `ROUND()` arredondar certinho em 2 casas.
- **`ORDER BY receita`** — `receita` não é uma coluna de nenhuma tabela, é o nome que a própria consulta deu pro resultado da soma (logo depois de `AS`). O PostgreSQL permite usar esse nome no `ORDER BY` da mesma consulta que o criou.

In [16]:
cursor.execute('''
    SELECT cat.category_name,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN products p ON p.product_id = od.product_id
    JOIN categories cat ON cat.category_id = p.category_id
    GROUP BY cat.category_name
    ORDER BY receita DESC
''')

for linha in cursor.fetchall():
    print(linha)

('Beverages', Decimal('267868.18'))
('Dairy Products', Decimal('234507.28'))
('Confections', Decimal('167357.23'))
('Meat/Poultry', Decimal('163022.36'))
('Seafood', Decimal('131261.74'))
('Condiments', Decimal('106047.08'))
('Produce', Decimal('99984.58'))
('Grains/Cereals', Decimal('95744.59'))


In [18]:
import pandas as pd

sala = """
SELECT cat.category_name,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN products p ON p.product_id = od.product_id
    JOIN categories cat ON cat.category_id = p.category_id
    GROUP BY cat.category_name
    ORDER BY receita DESC
"""

df = pd.read_sql_query(sala, conexao)
display(df)

C:\Users\s2bcl\AppData\Local\Temp\ipykernel_11516\3576670732.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sala, conexao)


,category_name,receita
0,Beverages,267868.18
1,Dairy Products,234507.28
2,Confections,167357.23
3,Meat/Poultry,163022.36
4,Seafood,131261.74
5,Condiments,106047.08
6,Produce,99984.58
7,Grains/Cereals,95744.59


## WHERE x HAVING — filtrando antes ou depois de agrupar

Os dois filtram linhas, mas em momentos diferentes da consulta:

| | Quando filtra | Pode usar `SUM()`, `COUNT()`, etc.? |
|---|---|---|
| `WHERE` | Antes do `GROUP BY` — filtra linhas individuais, cruas | Não |
| `HAVING` | Depois do `GROUP BY` — filtra grupos já agregados | Sim |

**`WHERE`** — quantos pedidos foram feitos depois de 01/01/1998 (filtra linhas de `orders`, sem agrupar nada):

In [19]:
cursor.execute("SELECT COUNT(*) FROM orders WHERE order_date > %s", ("1998-01-01",))
print("Pedidos depois de 1998-01-01:", cursor.fetchall())

Pedidos depois de 1998-01-01: [(267,)]


**`HAVING`** — quais categorias têm receita total acima de R$ 150.000. Isso é impossível de escrever com `WHERE`, porque `SUM(...)` só existe DEPOIS que as linhas já foram agrupadas por categoria — é a mesma consulta de receita por categoria de cima, só que agora filtrando o resultado já somado:

In [20]:
cursor.execute('''
    SELECT cat.category_name,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN products p ON p.product_id = od.product_id
    JOIN categories cat ON cat.category_id = p.category_id
    GROUP BY cat.category_name
    HAVING SUM(od.unit_price * od.quantity * (1 - od.discount)) > 150000
    ORDER BY receita DESC
''')

for linha in cursor.fetchall():
    print(linha)

('Beverages', Decimal('267868.18'))
('Dairy Products', Decimal('234507.28'))
('Confections', Decimal('167357.23'))
('Meat/Poultry', Decimal('163022.36'))


### Mais um relacionamento: receita por país

**Contextualização:** agora a diretoria quer saber quais países mais compram, pra decidir onde reforçar o time comercial. Mesma tabela fato (`order_details`), dimensão diferente (`customers`, através de `orders`).

In [21]:
cursor.execute('''
    SELECT c.country,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN orders o ON o.order_id = od.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    GROUP BY c.country
    ORDER BY receita DESC
    LIMIT 5
''')

for linha in cursor.fetchall():
    print(linha)

('USA', Decimal('245584.61'))
('Germany', Decimal('230284.63'))
('Austria', Decimal('128003.84'))
('Brazil', Decimal('106925.78'))
('France', Decimal('81358.32'))


### Criar coluna multiplicando colunas — o subtotal de cada venda

**Contextualização:** antes de somar tudo numa receita só por categoria ou por país, a diretoria também quer o detalhe linha a linha: quanto cada item vendido realmente valeu, sem juntar nada ainda.

A fórmula é a mesma multiplicação que você já usou pra somar receita (`unit_price * quantity * (1 - discount)`), só que agora sem `SUM()` e sem `GROUP BY` — cada linha do resultado ganha sua própria coluna nova, `subtotal`, calculada multiplicando 3 colunas que já existiam na tabela `order_details`. É a diferença entre **criar uma coluna nova a partir de outras, linha a linha** (o que este código faz) e **agregar essa coluna depois** (o que você já tinha feito antes, com `SUM()`).

In [22]:
subtotais = pd.read_sql('''
    SELECT od.order_id, p.product_name, od.unit_price, od.quantity, od.discount,
           ROUND((od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS subtotal
    FROM order_details od
    JOIN products p ON p.product_id = od.product_id
    ORDER BY subtotal DESC
    LIMIT 10
''', conexao)
display(subtotais)

C:\Users\s2bcl\AppData\Local\Temp\ipykernel_11516\117397323.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  subtotais = pd.read_sql('''


,order_id,product_name,unit_price,quantity,discount,subtotal
0,10981,Côte de Blaye,263.50,60,0.00,15810.00
1,10865,Côte de Blaye,263.50,60,0.05,15019.50
2,10417,Côte de Blaye,210.80,50,0.00,10540.00
3,10889,Côte de Blaye,263.50,40,0.00,10540.00
4,10897,Thüringer Rostbratwurst,123.79,80,0.00,9903.20
5,10353,Côte de Blaye,210.80,50,0.20,8432.00
6,10424,Côte de Blaye,210.80,49,0.20,8263.36
7,10817,Côte de Blaye,263.50,30,0.00,7905.00
8,10540,Côte de Blaye,263.50,30,0.00,7905.00
9,10816,Côte de Blaye,263.50,30,0.05,7509.75


Repare que `subtotal` não existia em nenhuma tabela do banco — ela nasceu ali mesmo, na consulta, da multiplicação de `unit_price`, `quantity` e `(1 - discount)`. É a mesma conta de sempre, só que mostrada linha a linha em vez de somada.

### Criar coluna com `CASE WHEN` — classificando sem multiplicar

**Contextualização:** o time de logística quer sinalizar rapidamente quais pedidos tiveram frete alto, pra revisar o contrato com a transportadora.

Nem toda coluna nova vem de uma conta — às vezes ela vem de uma **classificação**. O `CASE WHEN ... THEN ... ELSE ... END` funciona como um `if`/`elif`/`else` dentro do próprio SQL: testa as condições de cima pra baixo, e a primeira que for verdadeira decide o valor da coluna nova (aqui chamada `faixa_frete`) pra aquela linha; se nenhuma bater, o `ELSE` decide.

In [ ]:
fretes = pd.read_sql('''
    SELECT order_id, freight,
           CASE
               WHEN freight > 100 THEN 'Frete Alto'
               WHEN freight > 30 THEN 'Frete Médio'
               ELSE 'Frete Baixo'
           END AS faixa_frete
    FROM orders
    ORDER BY freight DESC
    LIMIT 25
''', conexao)
display(fretes)

C:\Users\s2bcl\AppData\Local\Temp\ipykernel_11516\339058818.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fretes = pd.read_sql('''


,order_id,freight,faixa_frete
0,10540,1007.64,Frete Alto
1,10372,890.78,Frete Alto
2,11030,830.75,Frete Alto
3,10691,810.05,Frete Alto
4,10514,789.95,Frete Alto
5,11017,754.26,Frete Alto
6,10816,719.78,Frete Alto
7,10479,708.95,Frete Alto
8,10983,657.54,Frete Alto
9,11032,606.19,Frete Alto


`faixa_frete` também é uma coluna que não existia antes — só que, diferente do `subtotal` de cima, ela não veio de uma multiplicação, veio de uma regra de decisão. Multiplicar colunas existentes e classificar com `CASE WHEN` são as duas formas mais comuns de criar coluna nova direto no SQL, antes mesmo de o dado chegar no Pandas.

### Mais um relacionamento: receita por vendedor (funcionário)

**Contextualização:** a diretoria de vendas quer saber qual funcionário gerou mais receita, pra decidir a bonificação de fim de ano. Mesma tabela fato de sempre (`order_details`), só que agora a dimensão é `employees` — ligada a ela não direto, mas através de `orders` (que guarda `employee_id`, o funcionário responsável por aquele pedido).

In [ ]:
cursor.execute('''
    SELECT e.first_name, e.last_name, 
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN orders o ON o.order_id = od.order_id
    JOIN employees e ON e.employee_id = o.employee_id
    GROUP BY e.first_name, e.last_name
    ORDER BY receita DESC
''')

for linha in cursor.fetchall():
    print(linha)

('Margaret', 'Peacock', Decimal('232890.85'))
('Janet', 'Leverling', Decimal('202812.84'))
('Nancy', 'Davolio', Decimal('192107.60'))
('Andrew', 'Fuller', Decimal('166537.76'))
('Laura', 'Callahan', Decimal('126862.28'))
('Robert', 'King', Decimal('124568.23'))
('Anne', 'Dodsworth', Decimal('77308.07'))
('Michael', 'Suyama', Decimal('73913.13'))
('Steven', 'Buchanan', Decimal('68792.28'))


### Mais um relacionamento: receita por fornecedor

**Contextualização:** o time de compras quer saber quais fornecedores pesam mais na receita da Northwind, pra priorizar a relação comercial com eles. Agora a dimensão é `suppliers`, ligada a `order_details` através de `products` (que guarda `supplier_id`, o fornecedor de cada produto).

In [27]:
cursor.execute('''
    SELECT s.company_name AS fornecedor, s.country AS pais_fornecedor,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN products p ON p.product_id = od.product_id
    JOIN suppliers s ON s.supplier_id = p.supplier_id
    GROUP BY s.company_name, s.country
    ORDER BY receita DESC
    LIMIT 5
''')

for linha in cursor.fetchall():
    print(linha)

('Aux joyeux ecclésiastiques', 'France', Decimal('153691.28'))
('Plutzer Lebensmittelgroßmärkte AG', 'Germany', Decimal('145372.40'))
('Gai pâturage', 'France', Decimal('117981.18'))
('Pavlova, Ltd.', 'Australia', Decimal('106459.78'))
("G'day, Mate", 'Australia', Decimal('65626.77'))


### Um JOIN com 6 tabelas de uma vez — auditando uma venda de ponta a ponta

**Contextualização:** a auditoria interna quer um relatório único pra conferir as vendas da Hanari Carnes: quem foi o cliente, qual funcionário atendeu, qual produto foi vendido e quem é o fornecedor desse produto — tudo numa consulta só.

O código abaixo junta 6 tabelas na mesma consulta: `order_details` (fato) + `orders`, `customers`, `employees`, `products` e `suppliers` (5 dimensões). Cada `JOIN` novo só precisa dizer como se conectar com UMA tabela já presente na consulta — o SQL monta o resto.

Uma novidade no `SELECT`: `e.first_name || ' ' || e.last_name` usa o operador `||`, que concatena (junta) texto no PostgreSQL — aqui, junta o primeiro e o último nome do funcionário numa coluna só, separados por um espaço.

In [28]:
cursor.execute('''
    SELECT c.company_name AS cliente,
           e.first_name || ' ' || e.last_name AS vendedor,
           p.product_name AS produto,
           s.company_name AS fornecedor,
           od.quantity
    FROM order_details od
    JOIN orders o ON o.order_id = od.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    JOIN employees e ON e.employee_id = o.employee_id
    JOIN products p ON p.product_id = od.product_id
    JOIN suppliers s ON s.supplier_id = p.supplier_id
    WHERE c.company_name = %s
    ORDER BY p.product_name
''', ("Hanari Carnes",))

for linha in cursor.fetchall():
    print(linha)

('Hanari Carnes', 'Steven Buchanan', 'Alice Mutton', 'Pavlova, Ltd.', 15)
('Hanari Carnes', 'Margaret Peacock', 'Carnarvon Tigers', 'Pavlova, Ltd.', 20)
('Hanari Carnes', 'Janet Leverling', 'Chartreuse verte', 'Aux joyeux ecclésiastiques', 42)
('Hanari Carnes', 'Margaret Peacock', 'Côte de Blaye', 'Aux joyeux ecclésiastiques', 5)
('Hanari Carnes', 'Andrew Fuller', 'Côte de Blaye', 'Aux joyeux ecclésiastiques', 4)
('Hanari Carnes', 'Nancy Davolio', 'Côte de Blaye', 'Aux joyeux ecclésiastiques', 60)
('Hanari Carnes', 'Janet Leverling', 'Filo Mix', "G'day, Mate", 12)
('Hanari Carnes', 'Andrew Fuller', 'Flotemysost', 'Norske Meierier', 9)
('Hanari Carnes', 'Nancy Davolio', 'Gnocchi di nonna Alice', 'Pasta Buttini s.r.l.', 20)
('Hanari Carnes', 'Janet Leverling', 'Gorgonzola Telino', 'Formaggi Fortini s.r.l.', 20)
('Hanari Carnes', 'Nancy Davolio', 'Gorgonzola Telino', 'Formaggi Fortini s.r.l.', 35)
('Hanari Carnes', 'Margaret Peacock', 'Gorgonzola Telino', 'Formaggi Fortini s.r.l.', 10)
('

## Limpeza de dados — nem todo dado real está pronto pra usar

Diferente de um exercício didático, o `northwind` tem casos reais que exigem decisão antes de analisar. Dois exemplos concretos deste banco:

1. **Pedidos sem `shipped_date`** — significa que o pedido ainda não foi despachado, não que o dado está "quebrado". Se você calculasse um tempo médio de entrega sem excluir essas linhas, o resultado sairia errado.
2. **Produtos descontinuados** (`discontinued = 1`) — ainda aparecem em pedidos antigos, mas não deveriam entrar numa análise de "catálogo ativo hoje".

In [29]:
cursor.execute("SELECT COUNT(*) FROM orders WHERE shipped_date IS NULL")
print("Pedidos ainda não despachados:", cursor.fetchall())

cursor.execute("SELECT order_id, order_date FROM orders WHERE shipped_date IS NULL ORDER BY order_id LIMIT 5")
print("Exemplos:", cursor.fetchall())

Pedidos ainda não despachados: [(21,)]
Exemplos: [(11008, datetime.date(1998, 4, 8)), (11019, datetime.date(1998, 4, 13)), (11039, datetime.date(1998, 4, 21)), (11040, datetime.date(1998, 4, 22)), (11045, datetime.date(1998, 4, 23))]


Antes de calcular uma média de tempo de entrega, o filtro `WHERE shipped_date IS NOT NULL` remove exatamente esses pedidos pendentes — sem isso, a métrica mistura "ainda não chegou" com "demorou muito".

**Sobre `shipped_date - order_date` no código abaixo:** no PostgreSQL, subtrair uma data de outra devolve direto o número de dias entre elas (aqui, um `AVG()` dessa diferença dá a média de dias até o envio) — não precisa converter nada antes, diferente de subtrair dois textos ou dois números de tipos diferentes.

In [ ]:
cursor.execute('''
    SELECT ROUND(AVG(shipped_date - order_date), 1) AS media_dias_entrega
    FROM orders
    WHERE shipped_date IS NOT NULL
''')
print("Média de dias até o envio (só pedidos já despachados):", cursor.fetchall())

E antes de contar quantos produtos existem "hoje" no catálogo, o filtro `discontinued = 0` remove os produtos que a Northwind já parou de vender:

In [ ]:
cursor.execute("SELECT COUNT(*) FROM products")
print("Total de produtos (com descontinuados):", cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM products WHERE discontinued = 0")
print("Produtos ativos no catálogo:", cursor.fetchall())

## Delete — encerrando o cadastro de teste

**Contextualização:** o cadastro da Squad Estúdio foi só um teste — hora de remover.

In [ ]:
cursor.execute("DELETE FROM customers WHERE customer_id = %s", ("SQUAD",))
conexao.commit()

cursor.execute("SELECT * FROM customers WHERE customer_id = %s", ("SQUAD",))
print("Deve ficar vazio:", cursor.fetchall())

cursor.close()
conexao.close()

## ✏️ Atividade Bônus 1 — Sua vez

**Contextualização:** o time de compras quer saber quais categorias de produto existem no catálogo, para decidir onde focar a próxima campanha.

**Comando:** conecte no banco `northwind` e consulte todas as linhas de `category_name` da tabela `categories`. Depois, feche a conexão.

In [ ]:
# escreva seu código aqui

## ✏️ Atividade Bônus 2 — Sua vez

**Contextualização:** a diretoria de RH quer avaliar a carga de trabalho da equipe de vendas: quantos itens de pedido (linhas de `order_details`) cada funcionário processou ao todo.

**Comando:** conecte no banco `northwind` e escreva um `JOIN` entre `order_details`, `orders` e `employees` que traga, por funcionário (`first_name`, `last_name`), o total de linhas processadas usando `COUNT(*)`, agrupando por funcionário e ordenando do maior para o menor. Feche a conexão ao final.

In [ ]:
# escreva seu código aqui

## ✏️ Atividade Bônus 3 — Sua vez

**Contextualização:** o time de compras quer sinalizar rapidamente quais fornecedores têm site cadastrado, pra priorizar o contato comercial por lá.

**Comando:** conecte no banco `northwind` e consulte `company_name` e `homepage` da tabela `suppliers`, criando uma coluna nova chamada `tem_site` com `CASE WHEN` — `'Sim'` quando `homepage IS NOT NULL`, `'Não'` caso contrário. Feche a conexão ao final.

In [ ]:
# escreva seu código aqui

### ✅ O que você fez neste bônus

- Conectou o Python ao PostgreSQL com `psycopg2` (a mesma lógica do `pyodbc`, biblioteca diferente).
- Usou parâmetros `%s` pra evitar SQL Injection, e entendeu por que isso não é a mesma coisa da formatação de texto do Python.
- Fez o CRUD completo (Create, Read ×2, Update, Delete) num banco relacional de verdade.
- Criou colunas novas direto no SQL: multiplicando colunas existentes (`subtotal` de cada venda) e classificando com `CASE WHEN` (`faixa_frete`).
- Rodou JOINs reais de várias tabelas usando alias, incluindo agregações (receita por categoria, receita por país, receita por vendedor, receita por fornecedor) — e chegou a juntar 6 tabelas numa única consulta.
- Entendeu a diferença entre **tabela fato** (`order_details`, os eventos/transações) e **tabelas de dimensão** (`customers`, `products`, `categories`, `employees`, `suppliers`, o contexto descritivo em volta de cada fato).
- Diferenciou `WHERE` (filtra linhas antes de agrupar) de `HAVING` (filtra grupos já agregados).
- Praticou limpeza de dados real: excluir pedidos ainda não despachados antes de calcular tempo de entrega, e excluir produtos descontinuados antes de contar o catálogo ativo.
- Traduziu tabelas e colunas do Northwind pra português com `ALTER TABLE ... RENAME`.

In [ ]:
import psycopg2

conexaoRenomear = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor2 = conexaoRenomear.cursor()

cursor2.execute("ALTER TABLE customers RENAME TO clientes")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN customer_id TO id_cliente")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN company_name TO nome_empresa")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN contact_name TO nome_contato")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN city TO cidade")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN country TO pais")

cursor2.execute("ALTER TABLE orders RENAME TO pedidos")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN order_id TO id_pedido")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN customer_id TO id_cliente")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN order_date TO data_pedido")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN shipped_date TO data_envio")

cursor2.execute("ALTER TABLE order_details RENAME TO itens_pedido")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN order_id TO id_pedido")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN product_id TO id_produto")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN unit_price TO preco_unitario")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN quantity TO quantidade")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN discount TO desconto")

cursor2.execute("ALTER TABLE products RENAME TO produtos")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN product_id TO id_produto")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN product_name TO nome_produto")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN category_id TO id_categoria")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN discontinued TO descontinuado")

cursor2.execute("ALTER TABLE categories RENAME TO categorias")
cursor2.execute("ALTER TABLE categorias RENAME COLUMN category_id TO id_categoria")
cursor2.execute("ALTER TABLE categorias RENAME COLUMN category_name TO nome_categoria")

conexaoRenomear.commit()
print("Tabelas e colunas renomeadas para português.")

Confirme rodando a mesma pergunta do JOIN lá de cima ("quais produtos a Hanari Carnes comprou"), agora só com nomes em português:

In [ ]:
cursor2.execute('''
    SELECT cli.nome_empresa, prod.nome_produto, ip.quantidade
    FROM clientes cli
    JOIN pedidos ped ON ped.id_cliente = cli.id_cliente
    JOIN itens_pedido ip ON ip.id_pedido = ped.id_pedido
    JOIN produtos prod ON prod.id_produto = ip.id_produto
    WHERE cli.nome_empresa = %s
    ORDER BY prod.nome_produto
    LIMIT 5
''', ("Hanari Carnes",))

print(cursor2.fetchall())

cursor2.close()
conexaoRenomear.close()

### ✅ O que você fez neste bônus

- Conectou o Python ao PostgreSQL com `psycopg2` (a mesma lógica do `pyodbc`, biblioteca diferente).
- Usou parâmetros `%s` pra evitar SQL Injection, e entendeu por que isso não é a mesma coisa da formatação de texto do Python.
- Fez o CRUD completo (Create, Read ×2, Update, Delete) num banco relacional de verdade.
- Rodou JOINs reais de várias tabelas usando alias, incluindo agregações (receita por categoria, receita por país).
- Entendeu a diferença entre **tabela fato** (`order_details`, os eventos/transações) e **tabelas de dimensão** (`customers`, `products`, `categories`, o contexto descritivo em volta de cada fato).
- Diferenciou `WHERE` (filtra linhas antes de agrupar) de `HAVING` (filtra grupos já agregados).
- Praticou limpeza de dados real: excluir pedidos ainda não despachados antes de calcular tempo de entrega, e excluir produtos descontinuados antes de contar o catálogo ativo.
- Traduziu tabelas e colunas do Northwind pra português com `ALTER TABLE ... RENAME`.